# Deep Learning for NLP

**Deep Learning** is a type of Machine Learning that uses **Neural Networks with multiple layers** to learn patterns from data.

Deep Learning can be used in NLP tasks such as:
- Text Classification
- Sentiment Analysis
- Question Answering

In this lab, we will build a **deep learning model** to classify Yelp restaurant reviews as:

- **0 = Negative**
- **1 = Positive**

<img src="https://drive.google.com/uc?export=view&id=1rUaLO1wEOcUZ10Kwd0nUvS3jQzrqoOcI" width="550">
<img src="https://drive.google.com/uc?export=view&id=1XKPTKwxmIpP5cZIkDajlai6aI9jUX-C8" width="400">

## 1. Load the Yelp Dataset

Each row contains:

- a restaurant review
- its sentiment label


In [8]:
import pandas as pd

df = pd.read_csv("07-yelp-dataset.txt", sep="\t", names=["review", "label"])

print("Number of reviews:", len(df))
df.head()

Number of reviews: 1000


,review,label
0,Wow... Loved this place.,1
1,Crust is not good.,0
2,Not tasty and the texture was just nasty.,0
3,Stopped by during the late May bank holiday of...,1
4,The selection on the menu was great and so wer...,1


In [9]:
print(df["label"].value_counts())

label
1    500
0    500
Name: count, dtype: int64


## 2. Preprocessing the Dataset

In [10]:
import re

def preprocess_text(text):

    # 1. Convert to lowercase
    text = text.lower()

    # 2. Remove punctuation and special characters
    text = re.sub(r'[^a-zA-Z\s]', '', text)

    # 3. Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()

    return text

df["review"] = df["review"].apply(preprocess_text)
df.head()

,review,label
0,wow loved this place,1
1,crust is not good,0
2,not tasty and the texture was just nasty,0
3,stopped by during the late may bank holiday of...,1
4,the selection on the menu was great and so wer...,1


## 3. Split the Data

We divide the dataset into:

- **80% training data** — used to train the model
- **20% testing data** — used to evaluate the model

`stratify=y` keeps the proportion of positive and negative reviews similar in both sets.

In [11]:
from sklearn.model_selection import train_test_split

X = df["review"]
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)

print("Training reviews:", len(X_train))
print("Testing reviews:", len(X_test))

Training reviews: 800
Testing reviews: 200


## 4. Convert Text to TF-IDF Vectors

A neural network cannot directly understand text.

**TF-IDF converts each review into a vector of numbers.**


In [12]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=1000, lowercase=True)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print("Training TF-IDF shape:", X_train_tfidf.shape)
print("Testing TF-IDF shape:", X_test_tfidf.shape)

Training TF-IDF shape: (800, 1000)
Testing TF-IDF shape: (200, 1000)


## 5. Convert the Data to PyTorch Tensors

PyTorch neural networks work with **tensors**.

In [13]:
#pip install torch --index-url https://download.pytorch.org/whl/cpu

In [14]:
import torch
import torch.nn as nn

torch.manual_seed(42)  # reproducible weight initialisation

# Convert TF-IDF data to PyTorch tensors
X_train_tensor = torch.FloatTensor(X_train_tfidf.toarray())
X_test_tensor = torch.FloatTensor(X_test_tfidf.toarray())

y_train_tensor = torch.FloatTensor(y_train.values).reshape(-1, 1)

## 6. Build a Simple Neural Network

Our network has:

1. **Input layer** — receives the TF-IDF features
2. **Hidden layer** — learns useful patterns
3. **ReLU activation** — adds non-linearity
4. **Output layer** — produces one value for binary classification

We do **not** put `Sigmoid` inside the model because `BCEWithLogitsLoss` applies it internally in a numerically stable way.

In [15]:
input_size = X_train_tensor.shape[1]
# torch.size([800, 1000])

model = nn.Sequential(
    nn.Linear(input_size, 64),   # Input layer
    nn.ReLU(),

    nn.Linear(64, 32),           # Hidden layer
    nn.ReLU(),

    nn.Linear(32, 1)             # Output layer
)

print(model)

Sequential(
  (0): Linear(in_features=1000, out_features=64, bias=True)
  (1): ReLU()
  (2): Linear(in_features=64, out_features=32, bias=True)
  (3): ReLU()
  (4): Linear(in_features=32, out_features=1, bias=True)
)


## 7. Choose the Loss Function and Optimizer

- **BCEWithLogitsLoss** is used for binary classification.
To measure the model’s mistakes.
- **Adam** is responsible for changing the model's weights so that the loss becomes smaller.

In [16]:
loss_function = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

## 8. Train the Neural Network

One **epoch** means the model has seen the full training dataset once.

For every epoch:

1. Make predictions
2. Calculate the loss
3. Calculate gradients with backpropagation
4. Update the weights

In [17]:
epochs = 10

for epoch in range(epochs):
    model.train()

    # 1. Forward pass
    outputs = model(X_train_tensor)

    # 2. Calculate loss
    loss = loss_function(outputs, y_train_tensor)

    # 3. Backpropagation (clear old gradients first)
    optimizer.zero_grad()
    loss.backward()

    # 4. Update model weights
    optimizer.step()

    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch + 1}/{epochs} - Loss: {loss.item():.4f}")

Epoch 5/10 - Loss: 0.6242
Epoch 10/10 - Loss: 0.3577


## 9. Evaluate the Model

The model outputs raw scores called **logits**.

We apply `sigmoid` to convert them to probabilities between 0 and 1.

- Probability **≥ 0.5 → Positive (1)**
- Probability **< 0.5 → Negative (0)**

In [18]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

model.eval()

with torch.no_grad():
    test = model(X_test_tensor)
    test_probabilities = torch.sigmoid(test)
    predictions = (test_probabilities >= 0.5).int().numpy().flatten()

accuracy = accuracy_score(y_test, predictions)

print(f"Test Accuracy: {accuracy:.3f}")

Test Accuracy: 0.755


## 10. Classification Report and Confusion Matrix

The classification report shows:

- **Precision**
- **Recall**
- **F1-score**

The confusion matrix shows how many reviews were classified correctly and incorrectly.

In [19]:
print("Classification Report:")
print(classification_report(y_test, predictions, target_names=["Negative", "Positive"]))

print("Confusion Matrix:")
print(confusion_matrix(y_test, predictions))

Classification Report:
              precision    recall  f1-score   support

    Negative       0.81      0.67      0.73       100
    Positive       0.72      0.84      0.77       100

    accuracy                           0.76       200
   macro avg       0.76      0.76      0.75       200
weighted avg       0.76      0.76      0.75       200

Confusion Matrix:
[[67 33]
 [16 84]]


## Task 1 – Represent Reviews Using Word2Vec

In the previous part of the lab, TF-IDF was used to convert each review into numerical features.
Now, use Word2Vec instead.

Requirements:
Tokenize the training and testing reviews.
Train a Word2Vec model using the training reviews.
Use:
vector_size = 200
window = 5
min_count = 1
sg = 1

Convert each review into a single vector.
Convert the resulting vectors into PyTorch tensors.


In [20]:
from gensim.models import Word2Vec
import numpy as np

# 1. Tokenize the reviews (they are already lowercased and cleaned of punctuation)
train_tokens = [review.split() for review in X_train]
test_tokens = [review.split() for review in X_test]

print("Example tokens:", train_tokens[0])

Example tokens: ['it', 'is', 'perfect', 'for', 'a', 'sitdown', 'family', 'meal', 'or', 'get', 'together', 'with', 'a', 'few', 'friends']


In [21]:
# 2. Train a Skip-Gram Word2Vec model on the TRAINING reviews only
w2v_model = Word2Vec(
    sentences=train_tokens,
    vector_size=200,
    window=5,
    min_count=1,
    sg=1,          # 1 = Skip-Gram
    workers=1,     # single worker + seed -> reproducible vectors
    seed=42,
    epochs=20      # a few more passes, since the corpus is very small
)

print("Vocabulary size:", len(w2v_model.wv))
print("Words most similar to 'good':", w2v_model.wv.most_similar("good", topn=5))

Vocabulary size: 1787
Words most similar to 'good': [('food', 0.9915541410446167), ('staff', 0.9908944964408875), ('very', 0.9906996488571167), ('clean', 0.989606499671936), ('service', 0.988591194152832)]


### Convert each review into a single vector

A review has many words, but the neural network needs **one fixed-size vector per review**.

We take the **average of the Word2Vec vectors** of all words in the review.
Words that were not seen during training (possible in the test set) are skipped. If none of the words are known, the review gets a vector of zeros.

In [22]:
def review_to_vector(tokens, model):
    vectors = [model.wv[word] for word in tokens if word in model.wv]
    if len(vectors) == 0:
        return np.zeros(model.vector_size)
    return np.mean(vectors, axis=0)

X_train_w2v = np.array([review_to_vector(tokens, w2v_model) for tokens in train_tokens])
X_test_w2v = np.array([review_to_vector(tokens, w2v_model) for tokens in test_tokens])

print("Training Word2Vec shape:", X_train_w2v.shape)
print("Testing Word2Vec shape:", X_test_w2v.shape)

Training Word2Vec shape: (800, 200)
Testing Word2Vec shape: (200, 200)


In [23]:
# 3. Convert the review vectors into PyTorch tensors
X_train_w2v_tensor = torch.FloatTensor(X_train_w2v)
X_test_w2v_tensor = torch.FloatTensor(X_test_w2v)

y_train_w2v_tensor = torch.FloatTensor(y_train.values).reshape(-1, 1)

print(X_train_w2v_tensor.shape, y_train_w2v_tensor.shape)
print(X_test_w2v_tensor.shape)

torch.Size([800, 200]) torch.Size([800, 1])
torch.Size([200, 200])


## Task 2 – Build a Different Neural Network

Use the Word2Vec vectors from Task 1 as the input to a new neural network.
Your network must contain:
An input layer
Three hidden layers
ReLU activation functions
One output neuron
Use the following architecture:

Input
   
   ↓

128 neurons
   
   ↓

64 neurons
   
   ↓

32 neurons
   
   ↓

1 output neuron

Train the model using:
, BCEWithLogitsLoss
, Adam optimizer
, Learning rate = 0.001
, 15 epochs.
Then calculate the test accuracy.

In [24]:
torch.manual_seed(42)

w2v_input_size = X_train_w2v_tensor.shape[1]   # 200

w2v_nn = nn.Sequential(
    nn.Linear(w2v_input_size, 128),  # Input -> hidden layer 1
    nn.ReLU(),

    nn.Linear(128, 64),              # Hidden layer 2
    nn.ReLU(),

    nn.Linear(64, 32),               # Hidden layer 3
    nn.ReLU(),

    nn.Linear(32, 1)                 # Output neuron (logit)
)

print(w2v_nn)

Sequential(
  (0): Linear(in_features=200, out_features=128, bias=True)
  (1): ReLU()
  (2): Linear(in_features=128, out_features=64, bias=True)
  (3): ReLU()
  (4): Linear(in_features=64, out_features=32, bias=True)
  (5): ReLU()
  (6): Linear(in_features=32, out_features=1, bias=True)
)


In [25]:
w2v_loss_function = nn.BCEWithLogitsLoss()
w2v_optimizer = torch.optim.Adam(w2v_nn.parameters(), lr=0.001)

### Train for 15 epochs

With `lr = 0.001`, updating the weights only once per epoch (the whole dataset at once) would give just 15 small updates, which is not enough to learn anything.
So inside every epoch we split the training data into **mini-batches of 32 reviews** using a `DataLoader`. The model still sees the full training set once per epoch, but its weights are updated 25 times per epoch instead of once.

In [26]:
from torch.utils.data import TensorDataset, DataLoader

train_loader = DataLoader(
    TensorDataset(X_train_w2v_tensor, y_train_w2v_tensor),
    batch_size=32,
    shuffle=True
)

epochs = 15

for epoch in range(epochs):
    w2v_nn.train()
    epoch_loss = 0.0

    for batch_X, batch_y in train_loader:
        # 1. Forward pass
        outputs = w2v_nn(batch_X)

        # 2. Calculate loss
        loss = w2v_loss_function(outputs, batch_y)

        # 3. Backpropagation
        w2v_optimizer.zero_grad()
        loss.backward()

        # 4. Update model weights
        w2v_optimizer.step()

        epoch_loss += loss.item() * len(batch_X)

    print(f"Epoch {epoch + 1}/{epochs} - Loss: {epoch_loss / len(train_loader.dataset):.4f}")

Epoch 1/15 - Loss: 0.6939
Epoch 2/15 - Loss: 0.6891
Epoch 3/15 - Loss: 0.6806
Epoch 4/15 - Loss: 0.6755
Epoch 5/15 - Loss: 0.6702
Epoch 6/15 - Loss: 0.6747
Epoch 7/15 - Loss: 0.6673
Epoch 8/15 - Loss: 0.6681
Epoch 9/15 - Loss: 0.6618
Epoch 10/15 - Loss: 0.6629
Epoch 11/15 - Loss: 0.6623
Epoch 12/15 - Loss: 0.6627
Epoch 13/15 - Loss: 0.6592
Epoch 14/15 - Loss: 0.6605
Epoch 15/15 - Loss: 0.6611


In [27]:
w2v_nn.eval()

with torch.no_grad():
    w2v_logits = w2v_nn(X_test_w2v_tensor)
    w2v_probabilities = torch.sigmoid(w2v_logits)
    w2v_predictions = (w2v_probabilities >= 0.5).int().numpy().flatten()

w2v_accuracy = accuracy_score(y_test, w2v_predictions)

print(f"Word2Vec Test Accuracy: {w2v_accuracy:.3f}")
print()
print("Classification Report:")
print(classification_report(y_test, w2v_predictions, target_names=["Negative", "Positive"]))
print("Confusion Matrix:")
print(confusion_matrix(y_test, w2v_predictions))

Word2Vec Test Accuracy: 0.635

Classification Report:
              precision    recall  f1-score   support

    Negative       0.68      0.52      0.59       100
    Positive       0.61      0.75      0.67       100

    accuracy                           0.64       200
   macro avg       0.64      0.64      0.63       200
weighted avg       0.64      0.64      0.63       200

Confusion Matrix:
[[52 48]
 [25 75]]


In [28]:
print(f"TF-IDF   + 2-hidden-layer network: {accuracy:.3f}")
print(f"Word2Vec + 3-hidden-layer network: {w2v_accuracy:.3f}")

TF-IDF   + 2-hidden-layer network: 0.755
Word2Vec + 3-hidden-layer network: 0.635


### Discussion

- **Word2Vec** gives dense 200-dimensional vectors that capture word meaning, while **TF-IDF** gives sparse 1000-dimensional vectors based on word counts.
- Here the Word2Vec model is trained on only **800 short reviews**, which is very little data for learning good embeddings. Averaging the word vectors also loses word order and dilutes strong sentiment words such as *"not"*, *"great"* or *"worst"*.
- TF-IDF keeps each sentiment word as its own feature, so on a small dataset like this it is usually as good as or better than self-trained Word2Vec. Word2Vec tends to win when it is trained on a large corpus (or when pre-trained embeddings such as GloVe or Google News vectors are used).